# Download các thư viện cần thiết


In [1]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Tạo hàm để đọc file parquet (đọc các file parquet - data sau khi được processed)

In [2]:
def read_parquet(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return user_chunk_df

In [3]:
def read_parquet_item(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    item_chunk_files = [file for file in files if 'item_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return item_chunk_df

In [4]:
def read_parquet_purchase(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return purchase_chunk_df

# Tạo hàm lưu file parquet sau mỗi task

In [5]:
def split_and_save_parquet(df, num_files, output_dir):
    """
    Tách DataFrame thành nhiều file Parquet và lưu vào thư mục đích.
    
    :param df: DataFrame cần tách
    :param num_files: Số lượng file Parquet muốn tách
    :param output_dir: Thư mục lưu các file Parquet
    """
    # Đảm bảo thư mục tồn tại
    os.makedirs(output_dir, exist_ok=True)
    
    # Tính số dòng mỗi file sẽ có
    num_rows = df.height
    rows_per_file = num_rows // num_files

    # Tách DataFrame thành các phần và lưu mỗi phần vào một file Parquet
    for i in range(num_files):
        start_row = i * rows_per_file
        # Đảm bảo phần cuối cùng sẽ chứa tất cả các dòng còn lại
        end_row = (i + 1) * rows_per_file if i < num_files - 1 else num_rows
        
        # Tách phần DataFrame
        split_df = df[start_row:end_row]
        
        # Lưu phần DataFrame vào file .parquet
        file_path = os.path.join(output_dir, f"sale_pers.purchase_history_daily_chunk_{i}.parquet")
        split_df.write_parquet(file_path)
        print(f"Đã lưu file: {file_path}")

# Load các dataframe cần thiết

In [6]:
purchase_df = read_parquet_purchase("./preprocessed-feature")
purchase_df

item_id,quantity,customer_id,created_date,location,price,log_price,discount_rate,channel,payment_bucket,time_between_purchases,month,seasonal_trend,product_engagement_level,avg_transaction_amount_per_purchase,segment_name,avg_cat_l1_per_purchase,segment_name_right
str,i32,i32,datetime[μs],i32,f64,f64,f64,str,str,duration[μs],i8,str,str,f64,str,f64,str
"""6498000000007""",1,6169196,2024-04-01 20:30:53.900,170,122500.0,11.715874,0.3,"""Android""","""cash""",356d 22h 29m 3s 520ms,4,"""Spring""","""High""",585752.88672,"""Trung cấp""",2.196429,"""Mua vừa"""
"""1237000000007""",4,5651273,2024-04-01 20:16:41.520,642,319000.0,12.67295,0.149333,"""Android""","""wallet""",12d 19h 39m 43s 667ms,4,"""Spring""","""High""",891633.777778,"""Trung cấp""",1.333333,"""Mua ít"""
"""4952000000001""",3,5345838,2024-04-01 20:26:26.210,342,625000.0,13.345509,0.0,"""In-Store""","""cash""",333d 13h 13m 43s 937ms,4,"""Spring""","""High""",999622.391545,"""Trung cấp""",1.466667,"""Mua vừa"""
"""5537000000014""",1,1803090,2024-04-26 19:34:00.987,443,75000.0,11.225257,0.0,"""In-Store""","""qr""",342d 17m 31s 84ms,4,"""Spring""","""High""",649447.582149,"""Trung cấp""",2.197183,"""Mua vừa"""
"""6548000000002""",1,903324,2024-04-26 18:20:57.147,228,35000.0,10.463132,0.0,"""In-Store""","""card""",351d 15m 53s 360ms,4,"""Spring""","""High""",452412.024172,"""Trung cấp""",2.482759,"""Mua vừa"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2700000000002""",2,7793664,2024-08-18 17:32:13.567,724,58500.0,10.976799,0.1,"""In-Store""","""cash""",133d 2h 24m 11s 7ms,8,"""Summer""","""High""",512092.216942,"""Trung cấp""",1.55,"""Mua vừa"""
"""2263000000017""",1,5189847,2024-08-18 19:31:07.177,473,130500.0,11.779136,0.1,"""In-Store""","""cash""",0µs,8,"""Summer""","""High""",117450.0,"""Bình dân""",1.0,"""Mua ít"""
"""4690000000001""",2,343695,2024-08-18 20:42:42.397,497,35000.0,10.463132,0.0,"""In-Store""","""card""",0µs,8,"""Summer""","""High""",97000.0,"""Bình dân""",1.0,"""Mua ít"""


In [7]:
item_df = read_parquet_item("./preprocessed-feature")
item_df.head()

item_id,price,category_l1,category,brand_final,target_user_group_final,item_type_final,color_final,origin_final,material_final,sale_status,description_merge,age_bucket_final,price_segment
str,"decimal[38,4]",str,str,str,str,str,str,str,str,i32,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Núm ty Dr Brown""","""Dr.Brown's""","""Sơ sinh""",null,"""Đen""","""Ý""","""Silicone""",0,"""Chi tiết sản phẩm …","""1-3M""","""Mid"""
"""0010290040150""",69000.0000,"""Thời trang""","""Bộ quần áo bé gái""","""Con Cưng""","""Bé Gái""","""Bộ quần áo""",null,null,null,0,null,"""2-4Y""","""Mid"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""Gặm nướu khác""","""Thương hiệu khác""","""Bé Trai""",null,"""Hồng""","""Đức""","""Silicone""",0,"""Chi tiết sản phẩm …",null,"""Low"""
"""0020010000094""",401000.0000,"""Tã""","""Merries_Sơ Sinh""","""Merries Nhật""","""Sơ sinh""",null,"""Đen""","""Nhật Bản, Nhật Bản""","""Giấy, bột giấy, vải không dệt,…",0,"""﻿﻿Tã dán Merries size S 82 miế…","""3-6M""","""High"""
"""0020010000098""",401000.0000,"""Tã""","""Merries_Tã Quần""","""Merries Nhật""","""Sơ sinh""",null,"""Đen""","""Nhật Bản, Nhật Bản""","""Giấy, bột giấy, vải không dệt,…",0,"""﻿﻿﻿Bỉm tã quần Merries size M …","""6-9M""","""High"""


In [8]:
user_df = read_parquet("./preprocessed-feature")
user_df.head()

customer_id,gender,location,province,membership,region,location_name,install_app,district,milk_segment_preference,diaper_segment_preference
i32,str,i32,str,str,str,str,str,str,i64,i64
8220125,"""Nữ""",240,"""Tiền Giang""","""Gold""","""Đồng bằng sông Cửu Long""","""TGI - 364-365 Nguyễn Huệ""","""In-Store""","""Gò Công""",null,null
8220124,"""Nữ""",996,"""Đồng Nai""","""Standard""","""Đông Nam Bộ""","""DON - 569 Quốc lộ 20""","""In-Store""","""Tân Phú""",null,null
8220138,"""Nữ""",606,"""Hồ Chí Minh""","""Standard""","""Đông Nam Bộ""","""HCM - 385 Bùi Đình Túy""","""In-Store""","""Bình Thạnh""",null,null
8220127,"""Nữ""",746,"""Hà Nội""","""Standard""","""Đồng bằng sông Hồng""","""HNI - 16B-4 Nguyễn Văn Lộc""","""In-Store""","""Hà Đông""",null,null
8220130,"""Nam""",418,"""Hồ Chí Minh""","""Standard""","""Đông Nam Bộ""","""HCM - 1069 Tỉnh Lộ 43""","""In-Store""","""Thủ Đức""",null,null


# Task A: Hãy thống kê những sản phẩm hay mua chung và số lần mua chung: item 1 | item 2 | #cooc

Chuyển dữ liệu của trường `created_date` sang dạng `timestamp` để việc xử lý trở nên dễ dàng hơn 

In [9]:
import polars as pl
from itertools import combinations
from collections import Counter

# --- Giả sử df_purchase có cột: customer_id, invoice_id, item_id ---
# df_purchase = pl.read_parquet("purchases.parquet")

# 1️⃣ Gom các sản phẩm trong cùng một hóa đơn
df_grouped = (
    purchase_df.group_by(["customer_id", "created_date"])
    .agg(pl.col("item_id").unique().alias("items"))
)

# 2️⃣ Đếm số lần xuất hiện cặp sản phẩm
cooc_counter = Counter()
for items in df_grouped["items"]:
    if len(items) > 1:
        for pair in combinations(sorted(items), 2):
            cooc_counter[pair] += 1

# 3️⃣ Kết quả
cooc_count = pl.DataFrame({
    "item_1": [i1 for (i1, i2) in cooc_counter.keys()],
    "item_2": [i2 for (i1, i2) in cooc_counter.keys()],
    "cooc_count": list(cooc_counter.values())
}).sort("cooc_count", descending=True)

print(cooc_count.head(10))

shape: (10, 3)
┌───────────────┬───────────────┬────────────┐
│ item_1        ┆ item_2        ┆ cooc_count │
│ ---           ┆ ---           ┆ ---        │
│ str           ┆ str           ┆ i64        │
╞═══════════════╪═══════════════╪════════════╡
│ 2803000000011 ┆ 2803000000013 ┆ 78235      │
│ 2803000000012 ┆ 2803000000013 ┆ 53478      │
│ 2803000000011 ┆ 2803000000012 ┆ 52843      │
│ 2803000000010 ┆ 2803000000012 ┆ 34204      │
│ 2803000000010 ┆ 2803000000013 ┆ 32656      │
│ 1371000000001 ┆ 1371000000002 ┆ 29677      │
│ 3880000000001 ┆ 3880000000002 ┆ 27793      │
│ 2803000000010 ┆ 2803000000011 ┆ 26985      │
│ 1371000000003 ┆ 1371000000006 ┆ 26776      │
│ 0029250010001 ┆ 0029250010003 ┆ 26509      │
└───────────────┴───────────────┴────────────┘


In [10]:
pairs = (
    pl.concat([
        cooc_count.select(
            pl.col("item_1").alias("item_id"),
            pl.col("item_2").alias("co_item"),
            pl.col("cooc_count")
        ),
        cooc_count.select(
            pl.col("item_2").alias("item_id"),
            pl.col("item_1").alias("co_item"),
            pl.col("cooc_count")
        )
    ])
)

# 🧠 Lấy top 5 co_item có cooc_count cao nhất cho từng item_id
top_co_items = (
    pairs.sort(["item_id", "cooc_count"], descending=[False, True])
         .group_by("item_id")
         .agg(pl.col("co_item").head(10).alias("top10_co_items"))
)

# 🔗 Gộp vào item_df
item_df = item_df.join(top_co_items, on="item_id", how="left")

# ✅ Kết quả
print(item_df.select(["item_id", "top10_co_items"]).head())


shape: (5, 2)
┌───────────────┬─────────────────────────────────┐
│ item_id       ┆ top10_co_items                  │
│ ---           ┆ ---                             │
│ str           ┆ list[str]                       │
╞═══════════════╪═════════════════════════════════╡
│ 0502020000004 ┆ ["0007150000143", "00200200002… │
│ 0010290040150 ┆ null                            │
│ 0008010000015 ┆ ["0008170000235", "22630000000… │
│ 0020010000094 ┆ ["5950000000001", "15120000000… │
│ 0020010000098 ┆ ["1512000000004", "00200200001… │
└───────────────┴─────────────────────────────────┘


Kiểm tra một vài số liệu của feature mới

In [11]:
target_id = "0502020000004"

top10_list = (
    item_df
    .filter(pl.col("item_id") == target_id)
    .select("top10_co_items")
    .item()   # chuyển giá trị trong cột thành object Python
)

print(top10_list)

shape: (10,)
Series: '' [str]
[
	"0007150000143"
	"0020020000264"
	"4553000000003"
	"1743000000004"
	"2707000000001"
	"4548000000003"
	"1396000000034"
	"4837000000001"
	"5358000000001"
	"0029110000055"
]


In [12]:
target_id = "0502020000004"

# 1️⃣ Lấy danh sách top 10 sản phẩm mua chung
top10_list = (
    item_df
    .filter(pl.col("item_id") == target_id)
    .select("top10_co_items")
    .item()  # chuyển list từ cột sang object Python
)

# 2️⃣ Lọc item_df để lấy thông tin chi tiết của các sản phẩm này
top10_info = (
    item_df
    .filter(pl.col("item_id").is_in(top10_list))
    .select(["item_id", "description_merge"])
)

# 3️⃣ In kết quả
print(top10_info["description_merge"])

shape: (10,)
Series: 'description_merge' [str]
[
	"Kẹo dẻo trái cây Chupa Chups 8…
	"Chi tiết sản phẩm             …
	"Chi tiết sản phẩm             …
	"Chi tiết sản phẩmTên sản phẩm:…
	"Chi tiết sản phẩm             …
	"Chi tiết sản phẩm             …
	"Bánh Ăn Sáng C'est Bon Orion S…
	"Kẹo trứng Kinder Joy cho bé g…
	"﻿Được làm từ gạo Japonica, bán…
	"﻿﻿Kẹo dẻo Huro 105g Pan Food v…
]


/tmp/ipykernel_2616730/3986287780.py:14: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col("item_id").is_in(top10_list))


In [13]:
# Lấy dữ liệu từ cột item_id và description_merge
item_ids = top10_info["item_id"].to_list()
descriptions = top10_info["description_merge"].to_list()

# In kết quả song song với item_id và description_merge
for item_id, description in zip(item_ids, descriptions):
    print(f"Item ID: {item_id}, Description: {description}\n")


Item ID: 2707000000001, Description: Kẹo dẻo trái cây Chupa Chups 8g với hương vị thơm ngon sẽ là món đồ ăn vặt siêu hấp dẫn đối với mọi lứa tuổi. Ưu điểm nổi bật Kẹo dẻo Chupa Chups nổi tiếng, dai ngon hấp dẫnHướng dẫn sử dụngĂn trực tiếp Hướng dẫn bảo quảnBảo quản nơi khô ráo, thoáng mát. Tránh ánh sáng. Thông tin chi tiết Tên sản phẩm: Kẹo dẻo trái cây Chupa Chups 8gThương hiệu: Chupa Chups Sản xuất tại: Việt Nam Trọng lượng: 8gĐối tượng sử dụng: Mọi lứa tuổi Thành phần: Đường, Siro Glucoza, chất điều chỉnh độ acid, hương trái cây giống tự nhiên, màu tổng hợp﻿

Item ID: 1396000000034, Description: Chi tiết sản phẩm                     Tên sản phẩm: Kẹo soda kèm hộp hình micro ca hát 15g Coris         Thương hiệu: CORIS         Xuất xứ: Nhật Bản         Trọng lượng: 15g                        Kẹo soda kèm hộp hình micro ca hát 15g là một loại kẹo giải trí dành cho trẻ em từ 3 tuổi của thương hiệu CORIS đến từ Nhật Bản. Sản phẩm không chỉ mang lại hương vị soda hấp dẫn mà còn cung cấp

# Task B. Hãy dự đoán tuổi của em bé dựa trên:
- Thông tin "age_group" của bảng item: Ngày đầu tiên mua

- Thông tin sữa có chữ "Step 1": Ngày đầu tiên mua

- Thông tin sữa có chữ "Mom": Ngày cuối cùng mua

Tạo bảng dự đoán: | customer_id | first_date_buy_step 1 | age_by_step1 | first_date_buy_age_group_0-3M | age_by_age_group | last_date_buy_milk4mom | age_by_milk4mom.

In [14]:
# 1️⃣ Lấy danh sách item_id
step1_items = item_df.filter(
    pl.col("description_merge").str.contains("(?i)Step 1") 
)["item_id"]
print(step1_items)

# Danh sách các từ khóa liên quan đến "milk4mom" (sữa cho mẹ)
milk4mom_keywords = [
    "milk for mom", "sữa cho mẹ", "sữa mẹ", "milk for breastfeeding", 
    "breast milk", "sữa bầu", "sữa dành cho mẹ", "nước uống cho mẹ"
]

# Tạo biểu thức lọc với các từ khóa (chuyển tất cả chuỗi thành chữ thường trước khi so sánh)
milk4mom_items = item_df.filter(
    pl.col("description_merge").str.to_lowercase().str.contains("|".join(milk4mom_keywords).lower())  # Chuyển thành chữ thường
)["item_id"]

# In kết quả
print(milk4mom_items)

# Lọc giá trị "0M" hoặc "1-3M" trong cột "age_bucket_final"
age_group_items = item_df.filter(pl.col("age_bucket_final").is_in(["0M", "1-3M"]))["item_id"]

print(age_group_items)

shape: (17,)
Series: 'item_id' [str]
[
	"1727000000001"
	"0006040000428"
	"2798000000001"
	"4697000000002"
	"6497000000002"
	…
	"6497000000017"
	"6497000000018"
	"5949000000025"
	"6679000000002"
	"6678000000004"
]
shape: (480,)
Series: 'item_id' [str]
[
	"0020010000438"
	"0020010000440"
	"0020020000051"
	"0006040000143"
	"0020010000289"
	…
	"4666000000004"
	"4667000000003"
	"6597000000001"
	"5194000000001"
	"5194000000002"
]
shape: (2_000,)
Series: 'item_id' [str]
[
	"0502020000004"
	"0020010000151"
	"0007040040001"
	"0007051040005"
	"0014570000020"
	…
	"4684000000003"
	"3389000000006"
	"3524000000153"
	"0502021160016"
	"6996000000174"
]


In [15]:
# 2️⃣ Nhóm theo từng điều kiện
# Sử dụng inner join để chỉ giữ lại customer_id có mặt trong purchase_df
step1_df = (
    purchase_df.filter(pl.col("item_id").is_in(step1_items))  # Lọc từ purchase_df theo item_id
    .group_by("customer_id")
    .agg(pl.col("created_date").min().alias("first_date_buy_step1"))
)
print(step1_df)

mom_df = (
    purchase_df.filter(pl.col("item_id").is_in(milk4mom_items))  # Lọc từ purchase_df theo item_id
    .group_by("customer_id")
    .agg(pl.col("created_date").max().alias("last_date_buy_milk4mom"))
)
print(mom_df)

age_group_df = (
    purchase_df.filter(pl.col("item_id").is_in(age_group_items))  # Lọc từ purchase_df theo item_id
    .group_by("customer_id")
    .agg(pl.col("created_date").min().alias("first_date_buy_age_group_0_3M"))
)
print(age_group_df)


/tmp/ipykernel_2616730/1299262212.py:4: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  purchase_df.filter(pl.col("item_id").is_in(step1_items))  # Lọc từ purchase_df theo item_id


shape: (168_074, 2)
┌─────────────┬─────────────────────────┐
│ customer_id ┆ first_date_buy_step1    │
│ ---         ┆ ---                     │
│ i32         ┆ datetime[μs]            │
╞═════════════╪═════════════════════════╡
│ 7134418     ┆ 2024-01-13 18:20:32.563 │
│ 7236661     ┆ 2024-07-10 15:45:40.800 │
│ 671786      ┆ 2024-03-24 09:53:02.387 │
│ 6464338     ┆ 2024-09-20 12:41:47.867 │
│ 2478984     ┆ 2024-09-15 14:48:55.493 │
│ …           ┆ …                       │
│ 6407317     ┆ 2024-02-07 12:13:53.333 │
│ 3825173     ┆ 2024-04-16 09:59:33.230 │
│ 7863353     ┆ 2024-09-12 08:27:32.553 │
│ 7499868     ┆ 2024-06-09 14:31:40.557 │
│ 7047380     ┆ 2024-05-11 21:17:29.130 │
└─────────────┴─────────────────────────┘


/tmp/ipykernel_2616730/1299262212.py:11: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  purchase_df.filter(pl.col("item_id").is_in(milk4mom_items))  # Lọc từ purchase_df theo item_id


shape: (995_566, 2)
┌─────────────┬─────────────────────────┐
│ customer_id ┆ last_date_buy_milk4mom  │
│ ---         ┆ ---                     │
│ i32         ┆ datetime[μs]            │
╞═════════════╪═════════════════════════╡
│ 6204144     ┆ 2024-11-17 10:10:44.223 │
│ 6011850     ┆ 2024-05-13 17:06:32.153 │
│ 5446371     ┆ 2024-01-11 18:11:27.993 │
│ 7635406     ┆ 2024-09-26 09:29:29.120 │
│ 5892961     ┆ 2024-10-14 11:55:28.350 │
│ …           ┆ …                       │
│ 1645707     ┆ 2024-10-31 09:41:55.007 │
│ 7978440     ┆ 2024-11-11 10:24:51.517 │
│ 7927058     ┆ 2024-10-04 20:53:44.583 │
│ 4345037     ┆ 2024-11-14 21:13:16.703 │
│ 7155783     ┆ 2024-12-30 15:11:57.680 │
└─────────────┴─────────────────────────┘


/tmp/ipykernel_2616730/1299262212.py:18: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  purchase_df.filter(pl.col("item_id").is_in(age_group_items))  # Lọc từ purchase_df theo item_id


shape: (1_162_810, 2)
┌─────────────┬───────────────────────────────┐
│ customer_id ┆ first_date_buy_age_group_0_3M │
│ ---         ┆ ---                           │
│ i32         ┆ datetime[μs]                  │
╞═════════════╪═══════════════════════════════╡
│ 6784435     ┆ 2024-09-07 21:29:36.877       │
│ 195965      ┆ 2024-05-25 19:31:08.167       │
│ 7924831     ┆ 2024-10-04 09:56:15.730       │
│ 8147404     ┆ 2024-12-11 11:32:38.153       │
│ 2487374     ┆ 2024-01-15 18:16:15.160       │
│ …           ┆ …                             │
│ 5879852     ┆ 2024-06-28 14:27:18.237       │
│ 6077651     ┆ 2024-02-08 17:33:22.960       │
│ 5796616     ┆ 2024-04-22 16:55:54.037       │
│ 1894629     ┆ 2024-01-04 16:06:11.470       │
│ 3164924     ┆ 2024-09-19 09:30:02.307       │
└─────────────┴───────────────────────────────┘


In [16]:
import polars as pl
from datetime import datetime

# 1️⃣ Join từng bước và xóa cột dư
pred_df = step1_df.join(age_group_df, on="customer_id", how="left")  # Left join để giữ tất cả customer_id từ purchase_df

# Nếu join tạo ra customer_id_right → drop
if "customer_id_right" in pred_df.columns:
    pred_df = pred_df.drop("customer_id_right")

# Join với bảng mom_df
pred_df = pred_df.join(mom_df, on="customer_id", how="left")  # Left join để giữ tất cả customer_id từ purchase_df

# Nếu join tạo ra customer_id_right → drop
if "customer_id_right" in pred_df.columns:
    pred_df = pred_df.drop("customer_id_right")

# 2️⃣ Tính toán ngày sinh dựa trên các cột (ngày đầu tiên mua sản phẩm hoặc ngày cuối mua sữa bầu)
pred_df = pred_df.with_columns([
    # age-group: ngày đầu tiên mua sản phẩm (ngày sinh)
    pl.when(pl.col("first_date_buy_age_group_0_3M").is_null())
    .then(pl.lit("1900-01-01"))
    .otherwise(pl.col("first_date_buy_age_group_0_3M"))
    .alias("birth_date_age_group"),

    # step 1: ngày đầu tiên mua sản phẩm cho trẻ sơ sinh (ngày sinh)
    pl.when(pl.col("first_date_buy_step1").is_null())
    .then(pl.lit("1900-01-01"))
    .otherwise(pl.col("first_date_buy_step1"))
    .alias("birth_date_step1"),

    # milk4mom: cộng thêm 1 tháng sau lần cuối mua sữa cho mẹ (ngày sinh giả định)
    pl.when(pl.col("last_date_buy_milk4mom").is_null())
    .then(pl.lit("1900-01-01"))
    .otherwise(
        pl.col("last_date_buy_milk4mom")
        .cast(pl.Date)  # Chuyển đổi thành kiểu Date (nếu cần)
        + pl.duration(days=30)  # Cộng thêm 30 ngày (1 tháng)
    )
    .alias("adjusted_milk4mom")  # Cộng thêm 1 tháng vào ngày mua sữa bầu
])


# 3️⃣ Tính độ tuổi (đơn vị: tháng)
# Để tính độ tuổi theo tháng, ta tính sự chênh lệch giữa ngày hiện tại và ngày sinh, rồi chia cho 30
# Lấy ngày hiện tại và chuyển đổi sang kiểu Datetime
current_date = pl.lit(datetime.now().strftime("%Y-%m-%d")).str.strptime(pl.Datetime, "%Y-%m-%d")

pred_df = pred_df.with_columns([
    # Tính tuổi theo step 1 từ ngày sinh (birth_date_step1)
    ((pl.col("birth_date_step1")
      .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S.%f", strict=False) - current_date)
      .dt.total_days() / 30).alias("age_by_step1_months"),

    # Tính tuổi theo age-group từ ngày sinh (birth_date_age_group)
    ((pl.col("birth_date_age_group")
      .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S.%f", strict=False) - current_date)
      .dt.total_days() / 30).alias("age_by_age_group_months"),

    # Tính tuổi cho milk4mom (giả sử sinh một tháng sau ngày cuối mua sữa)
    ((pl.col("adjusted_milk4mom")
      .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S.%f", strict=False) - current_date)
      .dt.total_days() / 30).alias("age_by_milk4mom_months")
])



# 4️⃣ Chọn cột và tạo bảng theo đúng định dạng yêu cầu
final_df = pred_df.select([
    "customer_id",
    "birth_date_step1",
    "age_by_step1_months",
    "birth_date_age_group",
    "age_by_age_group_months",
    "adjusted_milk4mom",
    "age_by_milk4mom_months"
])

# 5️⃣ Kết quả
print(final_df)


/tmp/ipykernel_2616730/1364165799.py:52: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
  .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S.%f", strict=False) - current_date)
/tmp/ipykernel_2616730/1364165799.py:57: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
  .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S.%f", strict=False) - current_date)
/tmp/ipykernel_2616730/1364165799.py:62: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chro

shape: (168_074, 7)
┌─────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ customer_id ┆ birth_date_s ┆ age_by_step ┆ birth_date_ ┆ age_by_age_ ┆ adjusted_mi ┆ age_by_milk │
│ ---         ┆ tep1         ┆ 1_months    ┆ age_group   ┆ group_month ┆ lk4mom      ┆ 4mom_months │
│ i32         ┆ ---          ┆ ---         ┆ ---         ┆ s           ┆ ---         ┆ ---         │
│             ┆ str          ┆ f64         ┆ str         ┆ ---         ┆ str         ┆ f64         │
│             ┆              ┆             ┆             ┆ f64         ┆             ┆             │
╞═════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 7134418     ┆ 2024-01-13   ┆ -22.366667  ┆ 2024-01-13  ┆ -22.366667  ┆ 2024-02-12  ┆ null        │
│             ┆ 18:20:32.563 ┆             ┆ 18:20:32.56 ┆             ┆             ┆             │
│             ┆ 000          ┆             ┆ 3000        ┆             

Kiểm tra một số thứ

In [17]:
# Tính số lượng null và tỷ lệ phần trăm của null trong các cột age_by_step1, age_by_age_group, age_by_milk4mom
null_counts = pred_df.select([
    (pl.col("age_by_step1_months").is_null().sum().alias("null_age_by_step1")),
    (pl.col("age_by_age_group_months").is_null().sum().alias("null_age_by_age_group")),
    (pl.col("age_by_milk4mom_months").is_null().sum().alias("null_age_by_milk4mom")),
])

# Tính tổng số dòng dữ liệu
total_count = pred_df.height

# In ra số lượng null và tỷ lệ phần trăm
null_counts = null_counts  # Chuyển sang Pandas để dễ dàng tính toán tỷ lệ phần trăm

for col in null_counts.columns:
    null_count = null_counts[col][0]
    percentage = (null_count / total_count) * 100
    print(f"{col}: Null Count = {null_count}, Percentage = {percentage:.2f}%")


null_age_by_step1: Null Count = 0, Percentage = 0.00%
null_age_by_age_group: Null Count = 12787, Percentage = 7.61%
null_age_by_milk4mom: Null Count = 168074, Percentage = 100.00%


# Thêm đặc trưng vào dataset ban đầu

In [19]:
# --- TASK A: JOIN co-occurrence feature vào purchase_df ---

purchase_df = purchase_df.join(
    item_df.select(["item_id", "top10_co_items"]),
    on="item_id",
    how="left"
)

split_and_save_parquet(purchase_df, 72, "./preprocessed-feature")

purchase_df.head()

Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_0.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_1.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_2.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_3.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_4.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_5.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_6.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_7.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_8.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_9.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_10.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_d

item_id,quantity,customer_id,created_date,location,price,log_price,discount_rate,channel,payment_bucket,time_between_purchases,month,seasonal_trend,product_engagement_level,avg_transaction_amount_per_purchase,segment_name,avg_cat_l1_per_purchase,segment_name_right,top10_co_items,top10_co_items_right
str,i32,i32,datetime[μs],i32,f64,f64,f64,str,str,duration[μs],i8,str,str,f64,str,f64,str,list[str],list[str]
"""6498000000007""",1,6169196,2024-04-01 20:30:53.900,170,122500.0,11.715874,0.3,"""Android""","""cash""",356d 22h 29m 3s 520ms,4,"""Spring""","""High""",585752.88672,"""Trung cấp""",2.196429,"""Mua vừa""","[""5950000000001"", ""6382000000004"", … ""0007090000252""]","[""5950000000001"", ""6382000000004"", … ""0007090000252""]"
"""1237000000007""",4,5651273,2024-04-01 20:16:41.520,642,319000.0,12.67295,0.149333,"""Android""","""wallet""",12d 19h 39m 43s 667ms,4,"""Spring""","""High""",891633.777778,"""Trung cấp""",1.333333,"""Mua ít""","[""1237000000008"", ""2803000000013"", … ""1237000000009""]","[""1237000000008"", ""2803000000013"", … ""1237000000009""]"
"""4952000000001""",3,5345838,2024-04-01 20:26:26.210,342,625000.0,13.345509,0.0,"""In-Store""","""cash""",333d 13h 13m 43s 937ms,4,"""Spring""","""High""",999622.391545,"""Trung cấp""",1.466667,"""Mua vừa""","[""2017000000035"", ""6768000000005"", … ""2017000000038""]","[""2017000000035"", ""6768000000005"", … ""2017000000038""]"
"""5537000000014""",1,1803090,2024-04-26 19:34:00.987,443,75000.0,11.225257,0.0,"""In-Store""","""qr""",342d 17m 31s 84ms,4,"""Spring""","""High""",649447.582149,"""Trung cấp""",2.197183,"""Mua vừa""","[""5537000000011"", ""5537000000007"", … ""7115000000004""]","[""5537000000011"", ""5537000000007"", … ""7115000000004""]"
"""6548000000002""",1,903324,2024-04-26 18:20:57.147,228,35000.0,10.463132,0.0,"""In-Store""","""card""",351d 15m 53s 360ms,4,"""Spring""","""High""",452412.024172,"""Trung cấp""",2.482759,"""Mua vừa""","[""1727000000002"", ""6548000000001"", … ""2803000000011""]","[""1727000000002"", ""6548000000001"", … ""2803000000011""]"


In [20]:
# --- TASK B: Join features dự đoán tuổi vào purchase_df ---

purchase_df = purchase_df.join(
    final_df,
    on="customer_id",
    how="left"
)

split_and_save_parquet(purchase_df, 72, "./preprocessed-feature")

purchase_df.head()

Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_0.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_1.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_2.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_3.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_4.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_5.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_6.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_7.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_8.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_9.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_daily_chunk_10.parquet
Đã lưu file: ./preprocessed-feature/sale_pers.purchase_history_d

item_id,quantity,customer_id,created_date,location,price,log_price,discount_rate,channel,payment_bucket,time_between_purchases,month,seasonal_trend,product_engagement_level,avg_transaction_amount_per_purchase,segment_name,avg_cat_l1_per_purchase,segment_name_right,top10_co_items,top10_co_items_right,birth_date_step1,age_by_step1_months,birth_date_age_group,age_by_age_group_months,adjusted_milk4mom,age_by_milk4mom_months
str,i32,i32,datetime[μs],i32,f64,f64,f64,str,str,duration[μs],i8,str,str,f64,str,f64,str,list[str],list[str],str,f64,str,f64,str,f64
"""6498000000007""",1,6169196,2024-04-01 20:30:53.900,170,122500.0,11.715874,0.3,"""Android""","""cash""",356d 22h 29m 3s 520ms,4,"""Spring""","""High""",585752.88672,"""Trung cấp""",2.196429,"""Mua vừa""","[""5950000000001"", ""6382000000004"", … ""0007090000252""]","[""5950000000001"", ""6382000000004"", … ""0007090000252""]","""2024-01-27 12:39:24.067000""",-21.9,"""2024-01-02 14:47:56.653000""",-22.733333,"""2025-01-23""",null
"""1237000000007""",4,5651273,2024-04-01 20:16:41.520,642,319000.0,12.67295,0.149333,"""Android""","""wallet""",12d 19h 39m 43s 667ms,4,"""Spring""","""High""",891633.777778,"""Trung cấp""",1.333333,"""Mua ít""","[""1237000000008"", ""2803000000013"", … ""1237000000009""]","[""1237000000008"", ""2803000000013"", … ""1237000000009""]",null,null,null,null,null,null
"""4952000000001""",3,5345838,2024-04-01 20:26:26.210,342,625000.0,13.345509,0.0,"""In-Store""","""cash""",333d 13h 13m 43s 937ms,4,"""Spring""","""High""",999622.391545,"""Trung cấp""",1.466667,"""Mua vừa""","[""2017000000035"", ""6768000000005"", … ""2017000000038""]","[""2017000000035"", ""6768000000005"", … ""2017000000038""]",null,null,null,null,null,null
"""5537000000014""",1,1803090,2024-04-26 19:34:00.987,443,75000.0,11.225257,0.0,"""In-Store""","""qr""",342d 17m 31s 84ms,4,"""Spring""","""High""",649447.582149,"""Trung cấp""",2.197183,"""Mua vừa""","[""5537000000011"", ""5537000000007"", … ""7115000000004""]","[""5537000000011"", ""5537000000007"", … ""7115000000004""]","""2024-01-31 12:21:11.130000""",-21.766667,"""2024-01-30 12:19:19.820000""",-21.8,"""2025-01-03""",null
"""6548000000002""",1,903324,2024-04-26 18:20:57.147,228,35000.0,10.463132,0.0,"""In-Store""","""card""",351d 15m 53s 360ms,4,"""Spring""","""High""",452412.024172,"""Trung cấp""",2.482759,"""Mua vừa""","[""1727000000002"", ""6548000000001"", … ""2803000000011""]","[""1727000000002"", ""6548000000001"", … ""2803000000011""]",null,null,null,null,null,null


# Zip file

In [22]:
import shutil

# Folder gốc bạn muốn zip
folder_path = "./preprocessed-feature"

# Tạo file zip: preprocessed-feature.zip
shutil.make_archive(
    base_name="preprocessed-feature",   # tên file zip đầu ra
    format="zip",
    root_dir=folder_path
)

print("Đã tạo file preprocessed-feature.zip")

Đã tạo file preprocessed-feature.zip
